In [7]:
from kloppy.domain import EventFactory
import inspect

event_factory = EventFactory()

print(type(event_factory))
print([m for m in dir(event_factory) if not m.startswith("_")])

<class 'kloppy.domain.services.event_factory.EventFactory'>
['build_ball_out', 'build_card', 'build_carry', 'build_clearance', 'build_duel', 'build_formation_change', 'build_foul_committed', 'build_generic', 'build_goalkeeper_event', 'build_interception', 'build_miscontrol', 'build_pass', 'build_player_off', 'build_player_on', 'build_pressure_event', 'build_recovery', 'build_shot', 'build_substitution', 'build_take_on']


In [8]:
from kloppy import footovision
from kloppy.config import get_config

dataset = footovision.load_event(
    event_data="kloppy/tests/files/footovision_events.json",
    meta_data="kloppy/tests/files/footovision_game.json",
    additional_metadata={},
    event_factory=event_factory,
)

In [9]:
player = dataset.metadata.teams[0].get_player_by_jersey_number(5)
player

<Player name='LEWIS DUNK' player_id='8569'>

In [10]:
player.positions

TimeContainer[PositionType]({'P1T00:00': <PositionType.LeftCenterBack: ('Left Center Back', 'LCB', 'CenterBack')>})

In [11]:
for start_time, end_time, position in player.positions.ranges():
    print(f"{start_time}:{end_time} - {position.code if position is not None else 'SUB'}")

P1T00:00:P2T52:14 - LCB


In [12]:
for player in dataset.metadata.teams[0].players:
    print(player)
    print(player.positions)

BART VERBRUGGEN
TimeContainer[PositionType]({'P1T00:00': <PositionType.Goalkeeper: ('Goalkeeper', 'GK', None)>})
LEWIS DUNK
TimeContainer[PositionType]({'P1T00:00': <PositionType.LeftCenterBack: ('Left Center Back', 'LCB', 'CenterBack')>})
JAN VAN HECKE
TimeContainer[PositionType]({'P1T00:00': <PositionType.RightCenterBack: ('Right Center Back', 'RCB', 'CenterBack')>})
GEORGINIO RUTTER
<TimeContainer>
YANKUBA MINTEH
TimeContainer[PositionType]({'P1T00:00': <PositionType.LeftAttackingMidfield: ('Left Attacking Midfield', 'LAM', 'AttackingMidfield')>})
JACK HINSHELWOOD
TimeContainer[PositionType]({'P1T00:00': <PositionType.AttackingMidfield: ('Attacking Midfield', 'AM', 'Midfielder')>})
CARLOS BALEBA
<TimeContainer>
DANNY WELBECK
TimeContainer[PositionType]({'P1T00:00': <PositionType.Striker: ('Striker', 'ST', 'Attacker')>})
JAMES MILNER
TimeContainer[PositionType]({'P1T00:00': <PositionType.RightDefensiveMidfield: ('Right Defensive Midfield', 'RDM', 'DefensiveMidfield')>})
KAORU MITOMA


In [13]:
len(dataset.records)

12743

In [14]:
df = dataset.to_df()

In [15]:
print(df.shape)
df.columns

(12743, 21)


Index(['event_id', 'event_type', 'period_id', 'timestamp', 'end_timestamp',
       'ball_state', 'ball_owning_team', 'team_id', 'player_id',
       'coordinates_x', 'coordinates_y', 'result', 'success',
       'end_coordinates_x', 'end_coordinates_y', 'receiver_player_id',
       'pass_type', 'set_piece_type', 'body_part_type', 'duel_type',
       'card_type'],
      dtype='object')

In [16]:
df['event_type'].value_counts()

event_type
GENERIC:generic    10424
PASS                 903
PRESSURE             570
CARRY                401
RECOVERY             113
DUEL                  92
BALL_OUT              84
CLEARANCE             56
TAKE_ON               40
SHOT                  28
FOUL_COMMITTED        22
CARD                  10
Name: count, dtype: int64

In [17]:
passes = df[df['event_type'] == 'PASS']
passes.head()

,event_id,event_type,period_id,timestamp,end_timestamp,ball_state,ball_owning_team,team_id,player_id,coordinates_x,...,result,success,end_coordinates_x,end_coordinates_y,receiver_player_id,pass_type,set_piece_type,body_part_type,duel_type,card_type
2,6742527193,PASS,1,0 days 00:00:00,0 days 00:00:03.280000,alive,78,78,8102,0.500,...,INCOMPLETE,False,0.103,0.000,6660,LONG_BALL,KICK_OFF,None,None,None
9,6742527195,PASS,1,0 days 00:00:26.600000,0 days 00:00:29.360000,alive,414,414,14584,0.242,...,INCOMPLETE,False,0.469,0.060,628,LONG_BALL,THROW_IN,None,None,None
14,6742527197,PASS,1,0 days 00:00:29.440000,0 days 00:00:30.560000,alive,78,78,8831,0.463,...,COMPLETE,True,0.398,0.034,8102,SIMPLE_PASS,None,None,None,None
19,6742527200,PASS,1,0 days 00:00:31.280000,0 days 00:00:31.920000,alive,78,78,8102,0.393,...,COMPLETE,True,0.385,0.138,8808,SIMPLE_PASS,None,None,None,None
20,6742527201,PASS,1,0 days 00:00:31.960000,0 days 00:00:32.360000,alive,78,78,8808,0.392,...,COMPLETE,True,0.430,0.136,8831,SIMPLE_PASS,None,None,None,None


In [18]:
passes.columns

Index(['event_id', 'event_type', 'period_id', 'timestamp', 'end_timestamp',
       'ball_state', 'ball_owning_team', 'team_id', 'player_id',
       'coordinates_x', 'coordinates_y', 'result', 'success',
       'end_coordinates_x', 'end_coordinates_y', 'receiver_player_id',
       'pass_type', 'set_piece_type', 'body_part_type', 'duel_type',
       'card_type'],
      dtype='object')

In [19]:
cat_columns = [
    'ball_state',
    'ball_owning_team',
    'team_id',
    'player_id',
    'result',
    'success',
    'pass_type',
    'set_piece_type',
    'body_part_type',
    'duel_type',
    'card_type',
]

In [20]:
for col in cat_columns:
    print('Column: ', col)
    print(passes[col].value_counts())

Column:  ball_state
ball_state
alive    903
Name: count, dtype: int64
Column:  ball_owning_team
ball_owning_team
78     481
414    422
Name: count, dtype: int64
Column:  team_id
team_id
78     481
414    422
Name: count, dtype: int64
Column:  player_id
player_id
8102     68
11573    62
8572     55
14584    50
8300     49
10668    47
21662    47
1999     46
8569     45
20601    44
8831     44
8808     43
19815    40
687      36
11284    32
8681     28
21778    27
8225     27
23676    26
628      16
6660     16
20182    15
1017     13
15848     8
29222     6
9329      3
14464     3
9145      2
16625     2
2015      2
4784      1
Name: count, dtype: int64
Column:  result
result
COMPLETE      705
INCOMPLETE    180
OUT            17
OFFSIDE         1
Name: count, dtype: int64
Column:  success
success
True     705
False    198
Name: count, dtype: int64
Column:  pass_type
pass_type
SIMPLE_PASS       610
LONG_BALL         100
THROUGH_BALL       55
HEAD_PASS          50
HIGH_PASS          26
CR

In [21]:
for event in df['event_type'].unique():
    events = df[df['event_type'] == event]

    print(f"Event: {event}")

    for col in cat_columns:
        print(f"Column: {col}")
        print(events[col].value_counts())

Event: GENERIC:generic
Column: ball_state
Series([], Name: count, dtype: int64)
Column: ball_owning_team
Series([], Name: count, dtype: int64)
Column: team_id
team_id
414    5532
78     4892
Name: count, dtype: int64
Column: player_id
player_id
20182    603
8572     598
23676    572
21662    552
8300     552
11573    552
8808     540
14584    533
20601    474
8102     465
628      453
21778    443
6660     421
19815    386
687      385
8225     373
8831     362
8569     312
11284    309
1999     287
29222    164
15848    158
4784     139
14464    130
9329     114
10668    104
1017     100
16625     90
8681      67
2015      43
9145      23
Name: count, dtype: int64
Column: result
Series([], Name: count, dtype: int64)
Column: success
Series([], Name: count, dtype: int64)
Column: pass_type
Series([], Name: count, dtype: int64)
Column: set_piece_type
Series([], Name: count, dtype: int64)
Column: body_part_type
Series([], Name: count, dtype: int64)
Column: duel_type
Series([], Name: count,

In [22]:
first_event = dataset.records[1]
print(type(first_event))
first_event.team

<class 'kloppy.domain.models.event.GenericEvent'>


Team(team_id='414', name='BRIGHTON', ground=home, starting_formation=<FormationType.FOUR_TWO_THREE_ONE: '4-2-3-1'>, formations=<TimeContainer>, players=[<Player name='BART VERBRUGGEN' player_id='10668'>, <Player name='LEWIS DUNK' player_id='8569'>, <Player name='JAN VAN HECKE' player_id='19815'>, <Player name='GEORGINIO RUTTER' player_id='9329'>, <Player name='YANKUBA MINTEH' player_id='21778'>, <Player name='JACK HINSHELWOOD' player_id='20182'>, <Player name='CARLOS BALEBA' player_id='14464'>, <Player name='DANNY WELBECK' player_id='628'>, <Player name='JAMES MILNER' player_id='687'>, <Player name='KAORU MITOMA' player_id='15848'>, <Player name='FERDI KADIOGLU' player_id='14584'>, <Player name='DIEGO GOMEZ' player_id='23676'>, <Player name='YASIN AYARI' player_id='16625'>, <Player name='MATS WIEFFER' player_id='21662'>, <Player name='PASCAL GROSS' player_id='8572'>, <Player name='JOEL VELTMAN' player_id='2015'>], coach=None)

In [23]:
dataset.metadata.teams[0]

Team(team_id='414', name='BRIGHTON', ground=home, starting_formation=<FormationType.FOUR_TWO_THREE_ONE: '4-2-3-1'>, formations=<TimeContainer>, players=[<Player name='BART VERBRUGGEN' player_id='10668'>, <Player name='LEWIS DUNK' player_id='8569'>, <Player name='JAN VAN HECKE' player_id='19815'>, <Player name='GEORGINIO RUTTER' player_id='9329'>, <Player name='YANKUBA MINTEH' player_id='21778'>, <Player name='JACK HINSHELWOOD' player_id='20182'>, <Player name='CARLOS BALEBA' player_id='14464'>, <Player name='DANNY WELBECK' player_id='628'>, <Player name='JAMES MILNER' player_id='687'>, <Player name='KAORU MITOMA' player_id='15848'>, <Player name='FERDI KADIOGLU' player_id='14584'>, <Player name='DIEGO GOMEZ' player_id='23676'>, <Player name='YASIN AYARI' player_id='16625'>, <Player name='MATS WIEFFER' player_id='21662'>, <Player name='PASCAL GROSS' player_id='8572'>, <Player name='JOEL VELTMAN' player_id='2015'>], coach=None)

In [24]:
for player in dataset.metadata.teams[0].players:
    print(player.starting_position)

Goalkeeper
Left Center Back
Right Center Back
Striker
Left Attacking Midfield
Attacking Midfield
Right Defensive Midfield
Striker
Right Defensive Midfield
Left Attacking Midfield
Left Back
Right Attacking Midfield
Attacking Midfield
Right Back
Left Defensive Midfield
Right Wing Back


In [25]:
import json

with open(
    "kloppy/tests/files/footovision_events.json", "r", encoding="utf-8"
) as f:
    event_data = json.load(f)

print(type(event_data))

<class 'list'>


In [26]:
dataset.metadata

Metadata(periods=[Period(id=1, start_timestamp=datetime.timedelta(0), end_timestamp=datetime.timedelta(seconds=3020, microseconds=240000), prev_period=None, next_period=Period(id=2, start_timestamp=datetime.timedelta(0), end_timestamp=datetime.timedelta(seconds=3134, microseconds=280000), prev_period=..., next_period=None)), Period(id=2, start_timestamp=datetime.timedelta(0), end_timestamp=datetime.timedelta(seconds=3134, microseconds=280000), prev_period=Period(id=1, start_timestamp=datetime.timedelta(0), end_timestamp=datetime.timedelta(seconds=3020, microseconds=240000), prev_period=None, next_period=...), next_period=None)], teams=[Team(team_id='414', name='BRIGHTON', ground=home, starting_formation=<FormationType.FOUR_TWO_THREE_ONE: '4-2-3-1'>, formations=<TimeContainer>, players=[<Player name='BART VERBRUGGEN' player_id='10668'>, <Player name='LEWIS DUNK' player_id='8569'>, <Player name='JAN VAN HECKE' player_id='19815'>, <Player name='GEORGINIO RUTTER' player_id='9329'>, <Player

In [27]:
import json

with open(
    "kloppy/tests/files/footovision_events.json", "r", encoding="utf-8"
) as f:
    event_data = json.load(f)

print(type(event_data))

<class 'list'>


In [28]:
print(type(event_data))

if isinstance(event_data, dict):
    print(event_data.keys())
elif isinstance(event_data, list):
    print(len(event_data))
    print(event_data[0])

<class 'list'>
12743
{'id': 6742539212, 'gameId': 28116, 'teamId': 78, 'playerId': 0, 'eventTypeIds': [734], 'halftime': 1, 'startTime': 0, 'endTime': 0, 'startX': 0.5, 'startY': 0, 'endX': 0.5, 'endY': 0, 'teamPhaseId': 6742539215, 'opponentPhaseId': 6742539216, 'sequenceId': 6742539214}


In [29]:
from kloppy import footovision
from kloppy.config import get_config

dataset = footovision.load_tracking(
    raw_data="kloppy/tests/files/footovision_tracking.json",
    meta_data="kloppy/tests/files/footovision_game.json",
)

KeyError: 'pitch_length'